# 03 · Strict exact match and confidence intervals

Recalculate pointwise exact binomial intervals from archived group counts. These are not simultaneous confidence intervals.

In [ ]:
from pathlib import Path
import pandas as pd

# Run from the repository root or its notebooks/ directory.
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
candidates = sorted(ROOT.rglob("retained_human_annotations_990.csv"))
if len(candidates) != 1:
    raise FileNotFoundError(
        f"Expected one revised analytical dataset, found {len(candidates)}. "
        "Extract the revised 99-item package into the repository and remove duplicate copies."
    )
DATA = candidates[0].parent
print("Repository:", ROOT)
print("Analysis directory:", DATA)

def load(name):
    return pd.read_csv(DATA / name, encoding="utf-8-sig")


In [ ]:
from scipy.stats import binomtest
ci = load("qwen_accuracy_exact_ci95.csv")
assert (ci["n"] == 99).all()
recomputed = ci.copy()
intervals = [binomtest(int(k),int(n)).proportion_ci(confidence_level=0.95,method="exact") for k,n in zip(ci["correct"],ci["n"])]
recomputed["recomputed_lower_pct"] = [100*x.low for x in intervals]
recomputed["recomputed_upper_pct"] = [100*x.high for x in intervals]
for saved, fresh in [("ci95_lower_percent","recomputed_lower_pct"),("ci95_upper_percent","recomputed_upper_pct")]:
    assert (recomputed[saved]-recomputed[fresh]).abs().max() < 1e-6
display(recomputed)
assert ci["correct"].sum() == 109
print(f"C1 counted once: 109/990 = {109/990:.2%}")
print(f"Execution-weighted: 175/1188 = {175/1188:.2%}")


In [ ]:
import matplotlib.pyplot as plt
ax = ci.plot(x="group",y="accuracy_percent",kind="bar",legend=False,figsize=(11,5),
    yerr=[ci["accuracy_percent"]-ci["ci95_lower_percent"],ci["ci95_upper_percent"]-ci["accuracy_percent"]],capsize=4)
ax.set(ylabel="Strict exact match (%)",xlabel="Condition–language", title="Strict exact match with pointwise 95% exact binomial intervals")
plt.tight_layout()
plt.show()


Direct string equality is not equivalent to human-assessed correctness. The two overall rates use different weighting schemes and must not be conflated.